# Bank Muscat Customer Review Exploratory Data Analysis

This notebook profiles the project-provided customer review dataset, checks basic data quality, and visualizes sentiment, ratings, categories, and review volume over time. It does not modify the source CSV or infer causality.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
data_candidates = [
    Path("data/bank_muscat_customer_reviews_5000.csv"),
    Path("../data/bank_muscat_customer_reviews_5000.csv"),
]
data_path = next((path for path in data_candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Run this notebook from the project root or notebooks directory.")

df = pd.read_csv(data_path, parse_dates=["review_date"])
print(f"Loaded {len(df):,} reviews from {data_path}")
df.head()

## Dataset shape and quality checks

In [ ]:
print("Shape:", df.shape)
display(df.dtypes.to_frame("data_type"))
display(df.isna().sum().to_frame("missing_values"))
print("Duplicate review IDs:", df["review_id"].duplicated().sum())
print("Rating values:", sorted(df["rating"].dropna().unique()))
display(df.describe(include="all").transpose())

## Sentiment and rating distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sentiment_order = [label for label in ["Positive", "Neutral", "Negative"] if label in df["sentiment"].unique()]
sns.countplot(data=df, x="sentiment", order=sentiment_order, ax=axes[0])
axes[0].set(title="Reviews by sentiment", xlabel="Sentiment", ylabel="Review count")
sns.countplot(data=df, x="rating", order=sorted(df["rating"].dropna().unique()), ax=axes[1])
axes[1].set(title="Reviews by rating", xlabel="Rating", ylabel="Review count")
plt.tight_layout()

## Category and channel breakdowns

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
df["category"].value_counts().sort_values().plot.barh(ax=axes[0])
axes[0].set(title="Reviews by category", xlabel="Review count", ylabel="Category")
df["channel"].value_counts().sort_values().plot.barh(ax=axes[1])
axes[1].set(title="Reviews by channel", xlabel="Review count", ylabel="Channel")
plt.tight_layout()

## Monthly review volume and sentiment

In [ ]:
dated = df.dropna(subset=["review_date"]).copy()
dated["year_month"] = dated["review_date"].dt.to_period("M").astype(str)
monthly = dated.groupby(["year_month", "sentiment"]).size().unstack(fill_value=0)
monthly.plot(figsize=(12, 5), marker="o")
plt.title("Monthly review volume by sentiment")
plt.xlabel("Month")
plt.ylabel("Review count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()